# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fmarryam70-ux/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Markdown cell:
"""
Unit of analysis: one row = one (client_id, content_id, report_date) daily
search performance record from fact_content_daily_performance.

Time window: mid-panel month, month=2026-03 (March 2026).
The final month (June 2026) is a sealed test month — never used to build label logic.

Target/proxy: same idea as the starter CSV's is_declining_label — a page is "declining"
if its trailing-30-day impressions dropped >20% vs the prior 30-day window, measured
using report_date windows ending at the end of March.

Excluded on purpose: content items with fewer than 30 total impressions across both
windows are excluded — too little signal to call a real trend vs noise.
"""

In [20]:
!pip install duckdb -q
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [21]:
schema_check = con.sql("""
    DESCRIBE SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    LIMIT 1
""").df()

print(schema_check.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [22]:
labels = con.sql("""
    WITH windows AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) as last_15,
               SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) as prev_15
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *,
        CASE WHEN prev_15 > 0 AND (last_15 - prev_15) / prev_15 < -0.20 THEN 1 ELSE 0 END as is_declining
    FROM windows
""").df()

print("Base rate (declining %):", labels['is_declining'].mean())
labels.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rate (declining %): 0.1498716196441556


,client_hash_id,content_hash_id,last_15,prev_15,is_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,711.0,429.0,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,39.0,18.0,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,60.0,89.0,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,793.0,628.0,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1490.0,1280.0,0


FEATURES: impressions_30d, avg_position_30d, clicks_30d, days_with_sessions, ga4_available_flag

LABEL: is_declining (impressions_30d dropped >20% vs prior 30d window — computed, not yet in raw table)

CONTEXT (grouping/joins only, never features): client_id, content_id

EXCLUDED: trend_pct / trend_direction equivalents (label source — never features);
raw report_date (only used to define windows, not as a feature itself)

In [23]:
features = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as impressions_30d,
        AVG(gsc_avg_position) as avg_position_30d,
        SUM(gsc_clicks) as clicks_30d,
        SUM(CASE WHEN ga4_sessions > 0 THEN 1 ELSE 0 END) as days_with_sessions,
        MAX(ga4_data_available) as ga4_available_flag
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-30'
    GROUP BY client_hash_id, content_hash_id
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_30d,avg_position_30d,clicks_30d,days_with_sessions,ga4_available_flag
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,76.0,3.903416,0.0,0.0,<NA>
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,583.0,4.521127,3.0,0.0,<NA>
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,791.0,4.842544,1.0,0.0,<NA>
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,77.0,9.131090,0.0,0.0,<NA>
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1811.0,1.905686,6.0,0.0,<NA>


# Markdown — one line per feature:
"""
1. impressions_30d — knowable at decision moment because it's a trailing 30-day
   sum computed as of today, no future data used.
2. avg_position_30d — knowable because it's an average over the same past window.
3. clicks_30d — same trailing-window logic, purely historical.
4. days_with_sessions — count of past days with activity, no forward-looking info.
5. ga4_available_flag — a static per-client/content flag, known before any prediction.
"""

In [24]:
leak_check = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) as impressions_first_half,
           SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) as impressions_second_half
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY client_hash_id, content_hash_id
""").df()

print("This second-half column leaks future info if used to predict decline in first half — deleting it now.")

This second-half column leaks future info if used to predict decline in first half — deleting it now.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
availability = con.sql("""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

availability

,total_rows,ga4_available_rows
0,9841378,413966.0


In [26]:
count_span = con.sql("""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date,
           COUNT(DISTINCT client_hash_id) as n_clients,
           COUNT(DISTINCT content_hash_id) as n_content
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

count_span

,row_count,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [27]:
count_span = con.sql("""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date,
           COUNT(DISTINCT client_hash_id) as n_clients,
           COUNT(DISTINCT content_hash_id) as n_content
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

count_span

,row_count,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [28]:
gsc_start_check = con.sql("""
    SELECT COUNT(*) as clients_starting_after_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
    WHERE gsc_data_start > '2026-03-01'
""").df()
gsc_start_check

,clients_starting_after_march
0,15


"""
This slice can never tell us: causal reasons for decline (only correlation),
performance for clients whose gsc_data_start is after March 2026 (they simply
don't appear), or true engagement for rows before ga4_data_available — those
are zero-filled, not truly zero.
"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.